 # Experiment 1 — Loss Function Comparison

 Development experiment:
 - Subject: 2
 - Channel: Fz
 - Time window: 1 s
 - CV: GroupKFold, 5 folds over 40 trials

 Conditions:
 - MSE
 - Negative Pearson correlation

 Primary unit of inference:
 - 40 held-out trials

 Frequency range:
 - 4–25 Hz (video-representable range)
 - Main bands: theta, alpha, beta

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, r'D:\Thesis\Codes\Source')

from face2eeg.stats import (
    summarize_trials,
    compare_methods_trial_level,
    comparison_to_dataframes,
)

from face2eeg.plotting import (
    plot_band_comparison,
    plot_mean_curve_with_ci,
)

from face2eeg.analysis import (
    setup,
    experiment_paths,
    load_result,
    validate_trial_alignment,
)

from face2eeg.core.config import SEED

%matplotlib inline

In [ ]:
BASE = r'D:\Thesis\Codes\Data\Experiments\Cross-Validation'

base = setup(BASE)
exp = experiment_paths(base, 'Loss-Functions')

RES_DIR = exp['results']
FIG_DIR = exp['figures']

os.makedirs(FIG_DIR, exist_ok=True)

TABLE_DIR = os.path.join(exp['results'], 'Analysis-Tables')
os.makedirs(TABLE_DIR, exist_ok=True)

N_BOOT = 10_000
N_PERM = 10_000

print("RES_DIR   :", RES_DIR)
print("FIG_DIR   :", FIG_DIR)
print("TABLE_DIR :", TABLE_DIR)

## 1. Load results

In [ ]:
r_mse = load_result(
    RES_DIR,
    subject=2,
    channel='Fz',
    variant='MSE',
)

r_neg = load_result(
    RES_DIR,
    subject=2,
    channel='Fz',
    variant='NegPearson',
)

print("MSE:")
print(r_mse.summary())

print("\nNegPearson:")
print(r_neg.summary())

## 2. Extract trial-level data + alignment check

In [ ]:
y_true_mse, y_pred_mse = r_mse.get_all_trials()
trial_ids_mse = r_mse.get_trial_indices()

y_true_neg, y_pred_neg = r_neg.get_all_trials()
trial_ids_neg = r_neg.get_trial_indices()

validate_trial_alignment(
    trial_ids_mse,
    trial_ids_neg,
    name_a="MSE",
    name_b="NegPearson",
)

validate_trial_alignment(
    trial_ids_mse,
    r_mse.get_trial_indices(),
    name_a="Trial IDs",
    name_b="MSE result",
)

if not np.allclose(y_true_mse, y_true_neg, equal_nan=True):
    raise ValueError("MSE and NegPearson do not use identical EEG targets.")

print("MSE:")
print("  y_true :", y_true_mse.shape)
print("  y_pred :", y_pred_mse.shape)

print("NegPearson:")
print("  y_true :", y_true_neg.shape)
print("  y_pred :", y_pred_neg.shape)

print("Trial IDs:", trial_ids_mse)

## 2. Trial-level summary

In [ ]:
overall_mse = summarize_trials(
    y_true_mse,
    y_pred_mse,
    seed=SEED,
    n_boot=N_BOOT
)

overall_neg = summarize_trials(
    y_true_neg,
    y_pred_neg,
    seed=SEED,
    n_boot=N_BOOT
)

# %%
# print("=== MSE ===")
# print(overall_mse)

# print("\n=== NegPearson ===")
# print(overall_neg)

## 3. Comparison statistics

In [ ]:
comp = compare_methods_trial_level(
    summary_A=overall_mse,
    summary_B=overall_neg,
    method_names=("MSE", "NegPearson"),
    seed=SEED,
    n_perm=N_PERM,
    n_boot=N_BOOT,
)

dfs = comparison_to_dataframes(comp)

In [ ]:
for name, df in dfs.items():
    print(f"\n{'=' * 20} {name} {'=' * 20}")
    display(df.round(3))

In [ ]:
for name, df in dfs.items():
    csv_path = os.path.join(
        TABLE_DIR,
        f"exp1_{name}.csv"
    )

    tex_path = os.path.join(
        TABLE_DIR,
        f"exp1_{name}.tex"
    )

    df.to_csv(csv_path, index=False)
    df.to_latex(
        tex_path,
        index=False,
        escape=False,
    )

    print(f"Saved: {csv_path}")
    print(f"Saved: {tex_path}")

## 4. Compact descriptive table

In [ ]:
summary_table = pd.DataFrame(
    {
        "MSE": pd.Series(overall_mse),
        "NegPearson": pd.Series(overall_neg),
    }
)

# Nested objects are not useful in the manuscript table.
drop_cols = [
    c for c in ["band_summary", "curves_summary"]
    if c in summary_table.index
]

summary_table = summary_table.drop(index=drop_cols)

summary_path = os.path.join(
    TABLE_DIR,
    "exp1_summary.csv"
)

summary_table.to_csv(summary_path)

display(summary_table.round(2))

## 3. Band comparison figure

In [ ]:
plot_band_comparison(
    {
        "a": overall_mse,
        "b": overall_neg,
    },
    {
        "a": "MSE",
        "b": "NegPearson",
    },
    layout="side",
    error="ci_boot",
    save_fig=True,
    file_name=os.path.join(
        FIG_DIR,
        "exp1_band_comparison.pdf",
    ),
)

In [ ]:
plot_mean_curve_with_ci(
    {
        "MSE": overall_mse,
        "NegPearson": overall_neg,
    },
    what="coh",
    freq_range=(4, 25),
    save_fig=True,
    file_name=os.path.join(
        FIG_DIR,
        "exp1_coh_mean_curve_with_ci.pdf",
    ),
)

In [ ]:
plot_mean_curve_with_ci(
    {
        "MSE": overall_mse,
        "NegPearson": overall_neg,
    },
    what="psd_pred",
    freq_range=(4, 25),
    save_fig=True,
    file_name=os.path.join(
        FIG_DIR,
        "exp1_psd_pred_mean_curve_with_ci.pdf",
    ),
)

In [ ]:
print("=== OVERALL COMPARISON ===")
display(dfs["overall"])

print("\n=== COHERENCE COMPARISON ===")
display(dfs["coherence"])

print("\n=== RBP ERROR COMPARISON ===")
display(dfs["rbp_error"])